# 第12章 图像轮廓（跟随代码）

> 《OpenCV轻松入门：面向Python》（李立宗）第 12 章跟随代码。**全量收录原书代码**（例 12.1~例 12.39 + 章内独立演示/参数演示），缺失例题已按书补全，代码以网页笔记 ch12.md 为准（含适配修正：OpenCV 4.x 的 `findContours` 只有**两个**返回值；`np.int0` 在 NumPy 2.0 已删除，取整改用 `astype(np.int32)`）。
> 原书本章素材图（contours.bmp、moments.bmp、hand.bmp 等）不在素材目录，例题统一改用绘图函数**构造演示图**顶替（见 12.1 的构造函数 cell）；原 notebook 自有的 picture7/8/9 演示保留为"独立演示"。
> `cv2.imshow` 需在**本地 Python 环境**运行（浏览器/Jupyter 内不弹窗）；运行前请确认**工作目录为素材根目录**（本文件夹），否则 `imread` 返回 None。


# 第十二章 图像轮廓

> 边缘检测找出的是一条条不连续的"边"，本章把边缘连接成整体——轮廓（contour），并在轮廓之上建立一整套几何分析工具：`findContours` 查找轮廓、`drawContours` 绘制轮廓；矩（moments）与 Hu 矩描述并比较形状；矩形、圆、椭圆、直线、三角形、逼近多边形"拟合"轮廓；凸包与凸缺陷刻画外形凹陷；最后提取宽高比、等效直径、极点、掩模均值等特征值。学完本章，一个白色目标就能被量化成"几个点、多大、什么形状、朝哪转"的一组数字。


## 12.1 查找并绘制轮廓

- `cv2.findContours()`：查找图像内的轮廓信息，按参数返回特定表示方式的轮廓（曲线）：

$$
contours,hierarchy=cv2.findContours(img,mode,method)
$$

- `cv2.drawContours()`：把查找到的轮廓绘制到图像上，可控制实心/空心、线条粗细、颜色等样式，可绘制全部轮廓也可仅绘制指定的轮廓：

$$
image=cv2.drawContours(image,contours,contourIdx,color,thickness)
$$

**注意**：OpenCV 4.x 中 `findContours` 只有 `contours, hierarchy` **两个返回值**（原书 3.x 的三值解包写法会报 `ValueError: too many values to unpack`）；输入必须是 **8 位单通道**二值图，且 OpenCV 从**黑色背景中查找白色对象**。


In [1]:
# cv_show 辅助函数定义（本地运行显示窗口用）
import cv2
import numpy as np 
import matplotlib.pyplot as plt

def cv_show(name,img):
   cv2.imshow(name,img)
   cv2.waitKey(0)
   cv2.destroyAllWindows()

### 12.1 本章通用演示图构造函数（顶替原书素材）

原书本章素材图（contours.bmp、moments.bmp、contours0.bmp、cc.bmp、hand.bmp、cs.bmp、loc3.jpg 等）不在素材目录，以下例题统一改用绘图函数（`rectangle`/`circle`/`fillPoly`）构造演示图顶替，注释里标出原书素材名。**请先运行本 cell 再跟后续例题**。


In [ ]:
# ===== 12.1 本章通用演示图构造函数（顶替原书素材） =====
import cv2
import numpy as np

def shapes_img():
    """圆 + 方 + 箭头三个白色对象（顶替 contours.bmp）"""
    o = np.zeros((240, 320, 3), np.uint8)
    cv2.circle(o, (60, 60), 35, (255, 255, 255), -1)              # 圆
    cv2.rectangle(o, (25, 130), (95, 200), (255, 255, 255), -1)   # 方块
    arrow = np.array([[[230, 40]], [[280, 90]], [[260, 90]], [[260, 200]],
                      [[200, 200]], [[200, 90]], [[180, 90]]], np.int32)
    cv2.fillPoly(o, [arrow], (255, 255, 255))                     # 箭头
    return o

def hole_img():
    """带孔矩形 + 独立小矩形（顶替原书层次示意图）"""
    o = np.zeros((300, 400, 3), np.uint8)
    cv2.rectangle(o, (50, 50), (249, 179), (255, 255, 255), -1)   # 外矩形
    cv2.rectangle(o, (100, 90), (179, 139), (0, 0, 0), -1)        # 中间挖孔
    cv2.rectangle(o, (300, 200), (379, 259), (255, 255, 255), -1) # 独立小矩形
    return o

def hand_img():
    """带凹口的五边形（顶替 hand.bmp，凹口顶点在 (150,150)）"""
    o = np.zeros((300, 400, 3), np.uint8)
    poly = np.array([[[50, 250]], [[50, 50]], [[150, 150]],
                     [[250, 50]], [[250, 250]]], np.int32)
    cv2.fillPoly(o, [poly], (255, 255, 255))
    return o

def quad_img(off=(0, 0), size=(300, 400)):
    """斜四边形（顶替 cc.bmp 等单轮廓素材），顶点 (100,50)(200,80)(180,140)(80,110)"""
    V = np.array([[[100, 50]], [[200, 80]], [[180, 140]], [[80, 110]]], np.int32)
    o = np.zeros((size[0], size[1], 3), np.uint8)
    cv2.fillPoly(o, [V + np.array(off)], (255, 255, 255))
    return o

def circ_img():
    """实心圆，圆心 (280,120)，半径 50"""
    o = np.zeros((300, 400, 3), np.uint8)
    cv2.circle(o, (280, 120), 50, (255, 255, 255), -1)
    return o


#### 独立演示 · 查找并绘制轮廓（picture7.png）

读入彩色图 → 灰度 → 阈值二值化 → `RETR_LIST` 查找全部轮廓 → 用 `drawContours` 把轮廓画到一张空白画布上（不是画在原图上）。流程与例 12.1/12.2 同源，素材与画法不同。


In [2]:
import cv2


img=cv2.imread("picture7.png")

h,w,g=img.shape

blank=np.zeros((h,w,3),np.uint8)


gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
ret,binary=cv2.threshold(gray,127,255,cv2.THRESH_BINARY)

cv_show("binary",binary)
contours,hierarchy=cv2.findContours(gray,cv2.RETR_LIST,cv2.CHAIN_APPROX_SIMPLE)

img=cv2.drawContours(blank,contours,-1,(0,0,255),1)

cv_show("img",img)

### 12.1.1 查找图像轮廓：findContours 函数

`contours, hierarchy = cv2.findContours(image, mode, method)`
- `image`：原始图像，必须是 **8 位单通道**图像，所有非零值被处理为 1、零值保持不变——一般先用阈值处理或边缘检测得到满意的二值图再传入；
- `mode`：轮廓检索模式，决定检测到的轮廓及其层次关系；
- `method`：轮廓的近似方法，决定如何表示轮廓（保留多少点）。


#### 演示 · 观察 contours 的属性

`contours` 实测为 tuple（旧版本返回 list，都按序列用下标访问），每个轮廓是 Numpy 的 ndarray，形状 `(点数, 1, 2)`，坐标是 **[列 行]** 形式；`contours[i][j]` 是第 i 个轮廓内的第 j 个点。


In [ ]:
# ===== 12.1.1 观察返回值 contours 的属性 =====
import cv2
import numpy as np

o = hole_img()                                # 带孔矩形 + 独立小矩形（构造函数见 12.1 节开头）
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)    # 转灰度
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)   # 二值化
contours, hierarchy = cv2.findContours(binary,
                                       cv2.RETR_CCOMP,          # 两级层次检索
                                       cv2.CHAIN_APPROX_SIMPLE) # 压缩冗余点
print(type(contours))          # 容器类型
print(type(contours[0]))       # 单个轮廓的类型
print(len(contours))           # 轮廓个数
for i in range(len(contours)):
    print(len(contours[i]), contours[i].shape)    # 每个轮廓的点数与形状
print(contours[0])             # 第 0 个轮廓的具体点


#### 演示 · 四种检索模式的 hierarchy

`hierarchy` 形状 `(1, 轮廓数, 4)`，每行 `[Next, Previous, First_Child, Parent]`（后一轮廓、前一轮廓、第 1 个子轮廓、父轮廓的索引，为空记 -1）；**层次结构由 mode 决定**：RETR_EXTERNAL 只检外轮廓，RETR_LIST 不建立父子关系，RETR_CCOMP 组织成两层，RETR_TREE 按嵌套建立完整等级树（本例只有两层嵌套，CCOMP 与 TREE 输出一致，多层嵌套时二者不同）。


In [ ]:
# ===== 12.1.1 四种检索模式的 hierarchy =====
img = hole_img()[:, :, 0]                     # 取 BGR 图第 0 个通道当灰度图
ret, binary = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
for name, mode in [('RETR_EXTERNAL', cv2.RETR_EXTERNAL),   # 只检测外轮廓
                   ('RETR_LIST',      cv2.RETR_LIST),      # 不建立等级关系
                   ('RETR_CCOMP',     cv2.RETR_CCOMP),     # 组织成两级层次
                   ('RETR_TREE',      cv2.RETR_TREE)]:     # 建立等级树
    c, h = cv2.findContours(binary, mode, cv2.CHAIN_APPROX_SIMPLE)
    print(name, '轮廓数 =', len(c))
    print(h)


#### 演示 · 两种近似方法的点数对比

`CHAIN_APPROX_NONE` 存储**所有**轮廓点（相邻两点坐标差不超过 1 个像素）；`CHAIN_APPROX_SIMPLE` 压缩水平、垂直、对角方向的冗余点，只保留端点——矩形只需 4 个角点。另有 `CHAIN_APPROX_TC89_L1`、`CHAIN_APPROX_TC89_KCOS` 两种 teh-Chin 链近似。


In [ ]:
# ===== 12.1.1 两种近似方法的点数对比 =====
cn, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)     # 存所有点
cp, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)   # 压缩
print('NONE   各轮廓点数 =', [len(c) for c in cn])
print('SIMPLE 各轮廓点数 =', [len(c) for c in cp])


### 12.1.2 绘制图像轮廓：drawContours 函数

`image = cv2.drawContours(image, contours, contourIdx, color[, thickness[, lineType[, hierarchy[, maxLevel[, offset]]]]])`
- `contourIdx`：要绘制的轮廓索引，**负数（通常 -1）表示绘制全部轮廓**；
- `thickness`：画笔粗细，**设为 -1 时绘制实心轮廓**（填充）；
- `maxLevel`：控制绘制层次深度，需与 `hierarchy` 参数配合使用；`offset`：轮廓偏移量。

**注意**：该函数直接在传入的 image 上**原地绘制**，执行后 image 不再是原图——原图还有其他用途时必须先 `copy()` 一份副本传入。


### 例 12.1 · 绘制所有轮廓

`contourIdx` 传 -1 绘制全部轮廓。注意**轮廓编号由算法内部扫描顺序决定**，不按"从上到下、从左到右"排列（本例下方的方块反而是 contours[0]）——定位特定轮廓前先打印 `boundingRect` 核对索引再使用。


In [ ]:
# ===== 例 12.1 绘制所有轮廓 =====
o = shapes_img()                                  # 原书：o = cv2.imread('contours.bmp')
cv2.imshow("original", o)
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary,
                                       cv2.RETR_EXTERNAL,       # 只检测外轮廓
                                       cv2.CHAIN_APPROX_SIMPLE)
for i, c in enumerate(contours):                  # 看看每条轮廓在哪
    print('轮廓%d boundingRect=%s 点数=%d' % (i, cv2.boundingRect(c), len(c)))
o = cv2.drawContours(o, contours, -1, (0, 0, 255), 5)   # -1 绘制全部，红色，粗细 5
cv2.imshow("result", o)
cv2.waitKey()
cv2.destroyAllWindows()


### 例 12.2 · 逐个显示轮廓

循环把 `contourIdx` 依次设为 0, 1, 2, …，每条轮廓配一张黑底画布（`np.zeros`）单独绘制，运行后每个窗口只有一条白色轮廓（原书为便于纸质书观察把轮廓画成白色）。


In [ ]:
# ===== 例 12.2 逐个显示轮廓 =====
o = shapes_img()                                  # 原书：contours.bmp
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary,
                                       cv2.RETR_EXTERNAL,
                                       cv2.CHAIN_APPROX_SIMPLE)
n = len(contours)
contoursImg = []
for i in range(n):
    temp = np.zeros(o.shape, np.uint8)            # 每条轮廓配一张黑底画布
    contoursImg.append(temp)
    contoursImg[i] = cv2.drawContours(contoursImg[i], contours, i,   # 只画第 i 条
                                      (255, 255, 255), 5)
    cv2.imshow("contours[" + str(i) + "]", contoursImg[i])
cv2.waitKey()
cv2.destroyAllWindows()


#### 独立演示 · 提取前景对象（picture8.jpg，例 12.3 同款流程）

实心轮廓当掩模：`drawContours` 的 `thickness=-1` 画出实心掩模，再与原图 `bitwise_and`——掩模外全黑、掩模内保留原像素，前景对象被抠出。本 cell 用自有素材 picture8.jpg 演示，例 12.3 用构造图。


In [249]:
#提取前景图像

img=cv2.imread('picture8.jpg')

gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)

ret,binary=cv2.threshold(gray,127,255,cv2.THRESH_BINARY)

contours,hierarchy=cv2.findContours(binary,cv2.RETR_LIST,cv2.CHAIN_APPROX_SIMPLE)

mask=np.zeros(img.shape,np.uint8)

mask=cv2.drawContours(mask,contours,-1,(255,255,255),-1)

loc=cv2.bitwise_and(img,mask)

print(len(contours))
cv_show('img',img)
cv_show('mask',mask)
cv_show('loc',loc)

362


### 例 12.3 · 提取前景对象

`thickness=-1` 绘制实心轮廓，得到"对象白、背景黑"的二值掩模 mask，再与原图按位与：掩模外像素全部清零、掩模内原样保留（原书素材 loc3.jpg 为小花照片，此处用构造图顶替）。


In [ ]:
# ===== 例 12.3 用实心轮廓提取前景 =====
o = shapes_img()                                  # 原书：o = cv2.imread('loc3.jpg')（小花照片）
cv2.imshow("original", o)
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary,
                                       cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
mask = np.zeros(o.shape, np.uint8)                # 与原图同大小的黑底掩模
mask = cv2.drawContours(mask, contours, -1, (255, 255, 255), -1)   # thickness=-1：实心
cv2.imshow("mask", mask)
loc = cv2.bitwise_and(o, mask)                    # 按位与：掩模外全黑，掩模内保留原像素
cv2.imshow("location", loc)
cv2.waitKey()
cv2.destroyAllWindows()


## 12.2 矩特征

`retval = cv2.moments(array[, binaryImage])`

比较两个轮廓最简单的方法是比较二者的**轮廓矩**——轮廓、图像、点集的全局特征（大小、位置、角度、形状）。返回 24 个字段的字典，分三类：
- 空间矩 `m00`…`m03` 系列：零阶矩 `m00` 含义直观，就是**轮廓面积**；
- 中心矩 `mu20`…`mu03`：减去均值获得**平移不变性**，可比较不同位置的对象；
- 归一化中心矩 `nu20`…`nu03`：再除以物体总尺寸获得**缩放不变性**。


#### 独立演示 · 各轮廓的矩与面积（picture9.jpg，例 12.4/12.5 同款流程）

对每个轮廓打印完整矩字典（24 个字段）与 `m00` 面积，并用黑底画布逐条画出轮廓；阈值取 137（自有素材按图调整）。


In [250]:
import cv2
import numpy as np

img=cv2.imread("picture9.jpg")
cv_show("original",img)

o=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)

ret,binary=cv2.threshold(o,137,255,cv2.THRESH_BINARY)

contours,hierarchy=cv2.findContours(binary,cv2.RETR_LIST,cv2.CHAIN_APPROX_SIMPLE)


n=len(contours)

contoursImg=[]

for i in range(n):
    temp=np.zeros(img.shape,np.uint8)
    contoursImg.append(temp)

    contoursImg[i]=cv2.drawContours(contoursImg[i],contours,i,(255,255,255),1)
    cv_show("contours"+str(i),contoursImg[i])
# 每个轮廓的矩特征
for i in range(n):
    print("轮廓"+str(i)+"的矩:",cv2.moments(contours[i]))
# 轮廓面积
for i in range(n):
    print("轮廓"+str(i)+"的面积:",cv2.moments(contours[i])["m00"])

轮廓0的矩: {'m00': 24915.0, 'm10': 24528817.5, 'm01': 13790452.5, 'm20': 24195961405.0, 'm11': 13576700486.25, 'm02': 7689541365.0, 'm30': 23914137597858.75, 'm21': 13392464637667.5, 'm12': 7570353473842.5, 'm03': 4318735323746.25, 'mu20': 47340576.25, 'mu11': 0.0, 'mu02': 56525906.25, 'mu30': 0.0, 'mu21': 0.0, 'mu12': 0.0, 'mu03': 0.0, 'nu20': 0.07626262626262627, 'nu11': 0.0, 'nu02': 0.09105960264900663, 'nu30': 0.0, 'nu21': 0.0, 'nu12': 0.0, 'nu03': 0.0}
轮廓1的矩: {'m00': 95853.0, 'm10': 42031540.5, 'm01': 24586294.5, 'm20': 19460299717.0, 'm11': 10781090138.25, 'm02': 6875823249.0, 'm30': 9436185921101.25, 'm21': 4991566877410.5, 'm12': 3015048494686.5, 'm03': 2055770721470.25, 'mu20': 1029469207.75, 'mu11': 0.0, 'mu02': 569438709.75, 'mu30': 0.0, 'mu21': 0.0, 'mu12': 0.0, 'mu03': 0.0, 'nu20': 0.11204744069912609, 'nu11': 0.0, 'nu02': 0.06197771587743732, 'nu30': 0.0, 'nu21': 0.0, 'nu12': 0.0, 'nu03': 0.0}
轮廓2的矩: {'m00': 59015.0, 'm10': 76926052.5, 'm01': 11301372.5, 'm20': 100773561551.6

### 例 12.4 · 提取一幅图像内各个轮廓的矩

`cv2.moments(contours[i])` 逐条取矩。对称对象两个方向的归一化二阶中心矩相等（方块与圆 nu20 = nu02），上下不对称的箭头则不等。


In [ ]:
# ===== 例 12.4 观察各轮廓的矩 =====
o = shapes_img()                                  # 原书：moments.bmp
cv2.imshow("original", o)
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
n = len(contours)
for i in range(n):
    m = cv2.moments(contours[i])                  # 第 i 条轮廓的矩
    print('轮廓%d: m00=%.1f m10=%.1f mu20=%.2f nu20=%.6f nu02=%.6f'
          % (i, m['m00'], m['m10'], m['mu20'], m['nu20'], m['nu02']))
print('各轮廓面积（m00）:')
for i in range(n):
    print('轮廓%d 的面积:%d' % (i, cv2.moments(contours[i])['m00']))
cv2.waitKey()
cv2.destroyAllWindows()


### 12.2.2 计算轮廓的面积：cv2.contourArea() 函数

$$
retval=cv2.contourArea(contour,oriented)
$$
- contour：轮廓
- oriented：是否考虑方向。为 True 时返回值带正负号（表示轮廓点是按顺时针还是逆时针排列）；默认 False，返回绝对值。

面积按轮廓点围成的**边界多边形**计算，不是像素计数。


#### 独立演示 · oriented=True 的带符号面积

对 picture9.jpg 的轮廓取 `cv2.contourArea(contours[i], oriented=True)`：返回值带正负号（负号表示轮廓点按逆时针排列），绝对值与默认结果一致。


In [251]:
for i in range(n):
    print("轮廓的面积:",cv2.contourArea(contours[i],oriented=True))

轮廓的面积: -24915.0
轮廓的面积: -95853.0
轮廓的面积: -59015.0


### 例 12.5 · 计算各个轮廓的面积

`cv2.contourArea(contours[i])`：71×71 的方块得 4900 = 70×70，圆得 3754（略小于理想值 πr²≈3848，多边形逼近圆）——面积按**边界多边形**算，不是像素计数。这些值与例 12.4 的 m00 完全相同。


In [ ]:
# ===== 例 12.5 计算面积 =====
o = shapes_img()                                  # 原书：contours.bmp
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
cv2.imshow("original", o)
n = len(contours)
contoursImg = []
for i in range(n):
    print("contours[" + str(i) + "]面积=", cv2.contourArea(contours[i]))
    temp = np.zeros(o.shape, np.uint8)
    contoursImg.append(temp)
    contoursImg[i] = cv2.drawContours(contoursImg[i], contours, i,
                                      (255, 255, 255), 3)
    cv2.imshow("contours[" + str(i) + "]", contoursImg[i])
cv2.waitKey()
cv2.destroyAllWindows()


### 例 12.6 · 筛选面积大于阈值的轮廓

`if cv2.contourArea(contours[i]) > 5000` 才显示——三条面积 4900、9101、3754 中只有轮廓 1（箭头，9101）入选，这是"按面积筛掉小噪声"的标准写法。


In [ ]:
# ===== 例 12.6 按面积筛选轮廓 =====
o = shapes_img()                                  # 原书：contours.bmp，阈值 15000
cv2.imshow("original", o)
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
n = len(contours)
contoursImg = []
for i in range(n):
    temp = np.zeros(o.shape, np.uint8)
    contoursImg.append(temp)
    contoursImg[i] = cv2.drawContours(contoursImg[i], contours, i,
                                      (255, 255, 255), 3)
    if cv2.contourArea(contours[i]) > 5000:       # 面积大于 5000 才显示
        cv2.imshow("contours[" + str(i) + "]", contoursImg[i])
cv2.waitKey()
cv2.destroyAllWindows()


### 12.2.3 计算轮廓的长度：cv2.arcLength() 函数

$$
retval=cv2.arcLength(curve,closed)
$$
- curve：轮廓
- closed：布尔值，表示曲线是否闭合（轮廓一般取 True）

返回轮廓的长度（周长），同样按边界多边形计算。


#### 独立演示 · 各轮廓的周长

对 picture9.jpg 的各条轮廓取 `cv2.arcLength(contours[i], True)`（True：封闭曲线）。


In [252]:
for i in range(n):
    print("轮廓长度：",cv2.arcLength(contours[i],True))

轮廓长度： 632.0
轮廓长度： 1252.0
轮廓长度： 1008.0


### 例 12.7 · 显示长度大于平均值的轮廓

先求各轮廓长度之和与平均值，再只显示 `arcLength > 平均值` 的轮廓——按周长筛选的套路与按面积筛选完全一致（原书素材 contours0.bmp 有 10 条轮廓，此处构造图为 3 条）。


In [ ]:
# ===== 例 12.7 按周长筛选轮廓 =====
# ---------- 读取及显示原始图像 ----------
o = hole_img()                                    # 原书：contours0.bmp（10 条轮廓）
cv2.imshow("original", o)
# ---------- 获取轮廓 ----------
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
# ---------- 计算各轮廓长度之和、平均长度 ----------
n = len(contours)
cntLen = []                                       # 存储各轮廓的长度
for i in range(n):
    cntLen.append(cv2.arcLength(contours[i], True))     # True：封闭曲线
    print("第" + str(i) + "个轮廓的长度:%.1f" % cntLen[i])
cntLenSum = np.sum(cntLen)                        # 长度之和
cntLenAvr = cntLenSum / n                         # 平均长度
print("轮廓的总长度为：%.1f" % cntLenSum)
print("轮廓的平均长度为：%.2f" % cntLenAvr)
# ---------- 显示长度超过平均值的轮廓 ----------
contoursImg = []
for i in range(n):
    temp = np.zeros(o.shape, np.uint8)
    contoursImg.append(temp)
    contoursImg[i] = cv2.drawContours(contoursImg[i], contours, i,
                                      (255, 255, 255), 3)
    if cv2.arcLength(contours[i], True) > cntLenAvr:
        cv2.imshow("contours[" + str(i) + "]", contoursImg[i])
cv2.waitKey()
cv2.destroyAllWindows()


## 12.3 Hu 矩

`hu = cv2.HuMoments(m)`：以 `cv2.moments()` 的返回值 m 为参数，返回 7 个 Hu 矩值。

Hu 矩是归一化中心矩的线性组合，图像经过**平移、缩放、旋转**后仍能保持不变，因此经常用 Hu 矩识别图像特征；更直观的比较用 `cv2.matchShapes()`（见例 12.10）。（原书两处把"Hu 矩"误写作"Hu 距"。）


### 例 12.8 · 验证第 0 个 Hu 矩的关系

验证 h0 = ν20 + ν02，差值为 0，关系成立。注意这里 `cv2.moments()` 的参数是**灰度图像**而不是轮廓——两种用法都合法。


In [ ]:
# ===== 例 12.8 验证 h0 = nu20 + nu02 =====
import cv2
o1 = shapes_img()
gray = cv2.cvtColor(o1, cv2.COLOR_BGR2GRAY)       # 原书直接对灰度图求矩
HuM1 = cv2.HuMoments(cv2.moments(gray)).flatten() # flatten() 展开成 7 个元素的数组
print("nu20 =", cv2.moments(gray)['nu20'])
print("nu02 =", cv2.moments(gray)['nu02'])
print("nu20+nu02 =", cv2.moments(gray)['nu20'] + cv2.moments(gray)['nu02'])
print("HuM1[0] =", HuM1[0])
print("Hu[0]-(nu02+nu20) =",
      HuM1[0] - (cv2.moments(gray)['nu20'] + cv2.moments(gray)['nu02']))


### 例 12.9 · 三幅不同图像的 Hu 矩比较

同一形状放大 2 倍再平移（g1 与 g2），前 3 个 Hu 矩的相对差仅约 0.01%；与形状完全不同的圆（g3）相比差出 3 个数量级。Hu 矩的值本身非常小（常见 10⁻³~10⁻¹⁸），直接看差值意义有限，实际比较用例 12.10 的 `matchShapes` 更方便。


In [ ]:
# ===== 例 12.9 三幅图的 Hu 矩比较 =====
import cv2
g1 = quad_img((100, 100))[:, :, 0]                        # 四边形
g2 = cv2.resize(quad_img(), None, fx=2, fy=2,
                interpolation=cv2.INTER_NEAREST)[:, :, 0] # 放大 2 倍并移位
g3 = circ_img()[:, :, 0]                                  # 圆
HuM1 = cv2.HuMoments(cv2.moments(g1)).flatten()
HuM2 = cv2.HuMoments(cv2.moments(g2)).flatten()
HuM3 = cv2.HuMoments(cv2.moments(g3)).flatten()
print("HuM1[:3] =", HuM1[:3])
print("HuM2[:3] =", HuM2[:3])
print("HuM3[:3] =", HuM3[:3])
print("HuM1-HuM2[:3] =", HuM1[:3] - HuM2[:3])
print("HuM1-HuM3[:3] =", HuM1[:3] - HuM3[:3])


### 例 12.10 · 形状匹配：matchShapes

`retval = cv2.matchShapes(contour1, contour2, method, parameter)`
- method：比较方法（CONTOURS_MATCH_I1/I2/I3，常用 I1，写 1 即可）；parameter 固定填 0；
- retval：相似度值，**越小说明两个对象越相似**，同一对象为 0。

函数内部先各自算好 Hu 矩再比较，参数可以是轮廓也可以是灰度图。


In [ ]:
# ===== 例 12.10 matchShapes =====
import cv2
gray1 = cv2.cvtColor(quad_img((100, 100)), cv2.COLOR_BGR2GRAY)          # 原形状
big = cv2.resize(quad_img(), None, fx=2, fy=2,
                 interpolation=cv2.INTER_NEAREST)                       # 放大 2 倍
gray2 = cv2.cvtColor(big, cv2.COLOR_BGR2GRAY)                           # 原书 cs2.bmp
gray3 = cv2.cvtColor(circ_img(), cv2.COLOR_BGR2GRAY)                    # 原书 cc.bmp
ret1, binary1 = cv2.threshold(gray1, 127, 255, cv2.THRESH_BINARY)
ret2, binary2 = cv2.threshold(gray2, 127, 255, cv2.THRESH_BINARY)
ret3, binary3 = cv2.threshold(gray3, 127, 255, cv2.THRESH_BINARY)
contours1, hierarchy = cv2.findContours(binary1, cv2.RETR_LIST,
                                        cv2.CHAIN_APPROX_SIMPLE)
contours2, hierarchy = cv2.findContours(binary2, cv2.RETR_LIST,
                                        cv2.CHAIN_APPROX_SIMPLE)
contours3, hierarchy = cv2.findContours(binary3, cv2.RETR_LIST,
                                        cv2.CHAIN_APPROX_SIMPLE)
cnt1, cnt2, cnt3 = contours1[0], contours2[0], contours3[0]
print("相同图像的matchShape=", cv2.matchShapes(cnt1, cnt1, 1, 0.0))
print("相似图像的matchShape=", cv2.matchShapes(cnt1, cnt2, 1, 0.0))
print("不相似图像的matchShape=", cv2.matchShapes(cnt1, cnt3, 1, 0.0))


## 12.4 轮廓拟合

计算轮廓时，可能并不需要实际的轮廓点，而只需要一个接近轮廓的近似对象。OpenCV 为每种拟合都提供了专门函数：

| 拟合 | 函数 |
| --- | --- |
| 矩形包围框（正置） | `cv2.boundingRect` |
| 最小包围矩形（旋转） | `cv2.minAreaRect` + `cv2.boxPoints` |
| 最小包围圆形 | `cv2.minEnclosingCircle` |
| 最优拟合椭圆 | `cv2.fitEllipse` |
| 最优拟合直线 | `cv2.fitLine` |
| 最小外包三角形 | `cv2.minEnclosingTriangle` |
| 逼近多边形 | `cv2.approxPolyDP` |

本节部分程序用到了绘图函数（`rectangle`、`circle`、`ellipse`、`line`、`polylines`），详见第 19 章。本节示例处理的都是单轮廓图的 `contours[0]`，若原图有多个轮廓，要控制索引指向特定轮廓。


### 12.4.1 矩形包围框：boundingRect

`retval = cv2.boundingRect(array)`：返回矩形边界左上角顶点坐标及宽、高，也可以写成 `x, y, w, h = cv2.boundingRect(array)` 的 4 返回值形式。参数 `array` 是灰度图像或轮廓；得到的是**边与坐标轴平行的正置**包围框。


### 例 12.11 · 观察 boundingRect 的两种返回值形式

同一个包围框：4 值解包与单个返回值（元组）形式结果一致 (80, 50, 121, 91)——斜四边形顶点横跨 80~200、纵跨 50~140，包围框把斜边也框进去。


In [ ]:
# ===== 例 12.11 boundingRect 的两种返回形式 =====
import cv2
o = quad_img()                                    # 原书：cc.bmp
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
x, y, w, h = cv2.boundingRect(contours[0])        # 四个返回值形式
print("x=", x); print("y=", y); print("w=", w); print("h=", h)
rect = cv2.boundingRect(contours[0])              # 单个返回值形式
print("rect=", rect)


### 例 12.12 · 用 drawContours 绘制矩形包围框

自己构造包围框的 4 个角点 `[[x,y]], [[x+w,y]], [[x+w,y+h]], [[x,y+h]]`，把它当成一条轮廓传给 `drawContours` 画出来。


In [ ]:
# ===== 例 12.12 drawContours 画包围框 =====
import cv2
import numpy as np
o = quad_img()
cv2.imshow("original", o)
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
x, y, w, h = cv2.boundingRect(contours[0])
brcnt = np.array([[[x, y]], [[x+w, y]], [[x+w, y+h]], [[x, y+h]]])   # 四个角点
cv2.drawContours(o, [brcnt], -1, (255, 255, 255), 2)                 # 当成轮廓画出来
cv2.imshow("result", o)
cv2.waitKey()
cv2.destroyAllWindows()


### 例 12.13 · 用 cv2.rectangle() 绘制包围框

更简单：直接给左上角 `(x, y)` 与右下角 `(x+w, y+h)` 两个角点即可。


In [ ]:
# ===== 例 12.13 cv2.rectangle 画包围框 =====
import cv2
o = quad_img()
cv2.imshow("original", o)
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
x, y, w, h = cv2.boundingRect(contours[0])
cv2.rectangle(o, (x, y), (x+w, y+h), (255, 255, 255), 2)   # 左上角与右下角
cv2.imshow("result", o)
cv2.waitKey()
cv2.destroyAllWindows()


### 12.4.2 最小包围矩形框：minAreaRect + boxPoints

`retval = cv2.minAreaRect(points)`：求面积**最小**的旋转矩形，返回（最小外接矩形的中心 (x,y)，(宽度,高度)，旋转角度）。返回值结构不符合 `drawContours` 的参数要求，必须先用 `points = cv2.boxPoints(box)` 转成 4 个角点。

**注意**：原书取整写 `np.int0(points)`，`np.int0` 在 NumPy 2.0 已被移除（报 AttributeError），改用 `np.round(points).astype(np.int32)`。


### 例 12.14 · 计算最小包围矩形框

返回的中心就是四个顶点的几何中心，尺寸对应四边形两条邻边的长度；取整后的 4 个角点与构造四边形时给的顶点一一对应——矩形随对象倾斜。


In [ ]:
# ===== 例 12.14 minAreaRect 与 boxPoints =====
import cv2
import numpy as np
o = quad_img()
cv2.imshow("original", o)
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
rect = cv2.minAreaRect(contours[0])
print("返回值rect:\n", rect)
points = cv2.boxPoints(rect)                       # 转成 4 个角点
print("转换后的points：\n", points)
points = np.round(points).astype(np.int32)         # 原书：np.int0(points)，NumPy 2.0 已删除
image = cv2.drawContours(o, [points], 0, (255, 255, 255), 2)
cv2.imshow("result", o)
cv2.waitKey()
cv2.destroyAllWindows()


### 12.4.3 最小包围圆形：minEnclosingCircle

`center, radius = cv2.minEnclosingCircle(points)`：通过迭代算法构造面积最小的包围圆，返回**浮点数**的圆心与半径，绘图前要取整。


### 例 12.15 · 构造最小包围圆形

圆轮廓的包围圆就是它自己：圆心 (280, 120)、半径 50.0001，取整后用 `cv2.circle` 画出。


In [ ]:
# ===== 例 12.15 minEnclosingCircle =====
import cv2
o = circ_img()                                    # 原书：cc.bmp
cv2.imshow("original", o)
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
(x, y), radius = cv2.minEnclosingCircle(contours[0])    # 返回浮点数
center = (int(x), int(y))                          # 绘图前取整
radius = int(radius)
cv2.circle(o, center, radius, (255, 255, 255), 2)  # 画圆
cv2.imshow("result", o)
cv2.waitKey()
cv2.destroyAllWindows()


### 12.4.4 最优拟合椭圆：fitEllipse

`retval = cv2.fitEllipse(points)`：返回 RotatedRect 类型的拟合椭圆外接矩形——其中心点、宽高、旋转角度正好对应椭圆的中心、轴长、旋转角度。

**注意**：输入轮廓**至少要 5 个点**。直边矩形用 `CHAIN_APPROX_SIMPLE` 压缩后只剩 4 个角点，传入会报 `There should be at least 5 points to fit the ellipse`，需改用 `CHAIN_APPROX_NONE` 获取完整轮廓。


### 例 12.16 · 构造最优拟合椭圆

返回值依次是椭圆中心 (140.06, 94.98)、轴长 (73.40, 128.24)、旋转角度 105.85；用 `cv2.ellipse(o, ellipse, (0,255,0), 3)` 可直接按返回值画出椭圆。


In [ ]:
# ===== 例 12.16 fitEllipse =====
import cv2
o = quad_img()
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
cv2.imshow("original", o)
ellipse = cv2.fitEllipse(contours[0])
print("ellipse=", ellipse)
cv2.ellipse(o, ellipse, (0, 255, 0), 3)            # 按 fitEllipse 的返回值画椭圆
cv2.imshow("result", o)
cv2.waitKey()
cv2.destroyAllWindows()


### 12.4.5 最优拟合直线：fitLine

`line = cv2.fitLine(points, distType, param, reps, aeps)`：拟合使输入点到拟合直线的距离之和最小，距离如何度量由 `distType` 决定（DIST_L1 / DIST_L2 / DIST_HUBER 等）；`param` 设 0 时函数自动选最优值，`reps`（径向精度）、`aeps`（角度精度）通常 0.01。返回 `[vx, vy, x, y]`——单位方向向量与直线上一点。


### 例 12.17 · 构造最优拟合直线并画到图像两端

用点斜式把直线延拓到图像左右两条边界（lefty / righty），再整条 `cv2.line` 画出。实测方向向量 (0.9645, 0.264) 表示直线向右下方倾斜，过点 (140.10, 95.00)（≈质心）。


In [ ]:
# ===== 例 12.17 fitLine =====
import cv2
o = quad_img()
cv2.imshow("original", o)
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
rows, cols = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY).shape[:2]   # 高、宽
[vx, vy, x, y] = cv2.fitLine(contours[0], cv2.DIST_L2, 0, 0.01, 0.01)
lefty = int((-x*vy/vx) + y)          # 直线与左边界 x=0 的交点 y
righty = int(((cols-x)*vy/vx) + y)   # 直线与右边界 x=cols 的交点 y
cv2.line(o, (cols-1, righty), (0, lefty), (0, 255, 0), 2)
cv2.imshow("result", o)
cv2.waitKey()
cv2.destroyAllWindows()


### 12.4.6 最小外包三角形：minEnclosingTriangle

`retval, triangle = cv2.minEnclosingTriangle(points)`：返回最小外包三角形的面积与三个顶点集。三个顶点连接时用 `(i+1) % 3` 让第 3 个顶点连回第 1 个，三角形才闭合；顶点是 float32，部分版本的绘图函数不接受浮点坐标，先 `astype(int)` 更稳。


### 例 12.18 · 构造最小外包三角形

打印面积 area 与顶点集 trgl，再按"第 i 个顶点连第 (i+1)%3 个顶点"依次画三条边。


In [ ]:
# ===== 例 12.18 minEnclosingTriangle =====
import cv2
o = quad_img()
cv2.imshow("original", o)
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
area, trgl = cv2.minEnclosingTriangle(contours[0])
print("area=", area)
print("trgl:", trgl)
for i in range(0, 3):
    cv2.line(o, tuple(trgl[i][0].astype(int)),                 # 第 i 个顶点
             tuple(trgl[(i + 1) % 3][0].astype(int)),          # 下一个顶点，%3 让首尾相连
             (255, 255, 255), 2)
cv2.imshow("result", o)
cv2.waitKey()
cv2.destroyAllWindows()


### 12.4.7 逼近多边形：approxPolyDP

`approxCurve = cv2.approxPolyDP(curve, epsilon, closed)`：采用 **Douglas-Peucker 算法（DP 算法）**——先找到轮廓上距离最远的两点相连，再不断"找离当前直线最远的点加入"，直到轮廓上所有点到当前多边形的距离都小于 `epsilon`。epsilon 越小，多边形越贴近原轮廓；通常把 epsilon 设为**轮廓周长的百分比**（如 `0.02*cv2.arcLength(cnt, True)`）。


### 例 12.19 · 用不同精度逼近同一轮廓

epsilon 取周长 1% 以上时都收敛到 4 个顶点（坐标即构造时的四个顶点），降到 0.1% 才保留 124 个点——**epsilon 越小多边形越贴近原轮廓，也越失去"简化"的意义**。注意 `drawContours` 原地修改，每轮循环要在副本 adp 上画。


In [ ]:
# ===== 例 12.19 不同 epsilon 的逼近多边形 =====
import cv2
o = quad_img()                                    # 原书：cc.bmp
cv2.imshow("original", o)
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
L = cv2.arcLength(contours[0], True)              # 周长 361.42
print('原始轮廓点数 =', len(contours[0]))          # 204
for k in [0.1, 0.09, 0.055, 0.05, 0.02, 0.01, 0.001]:
    adp = o.copy()                                # drawContours 会改图，用副本
    epsilon = k * cv2.arcLength(contours[0], True)
    approx = cv2.approxPolyDP(contours[0], epsilon, True)
    print('epsilon = %.3f*周长 → 顶点数 = %d' % (k, len(approx)))
    adp = cv2.drawContours(adp, [approx], 0, (0, 0, 255), 2)
    cv2.imshow("result" + str(k), adp)
cv2.waitKey()
cv2.destroyAllWindows()


## 12.5 凸包

逼近多边形是轮廓的高度近似；有时更希望用一个**凸包**来简化它。凸包跟逼近多边形很像，但它是物体最外层的"凸"多边形：完全包含原有轮廓，且仅由轮廓上的点构成，任意连续三个点的内角小于 180°。最外层凸多边形与轮廓之间的部分称为**凸缺陷**（Convexity Defect），可用于手势识别等问题（指缝就是典型的凸缺陷）。


### 12.5.1 获取凸包：convexHull

`hull = cv2.convexHull(points[, clockwise[, returnPoints]])`
- `clockwise`：True 时凸包角点按顺时针排列，False 按逆时针；
- `returnPoints`：默认 True 返回凸包角点的 x/y 坐标；**False 时返回轮廓中凸包角点的索引**（索引形式是给 convexityDefects 用的）。


### 例 12.20 · returnPoints 的两种取值

方块轮廓本身只有 4 个点，凸包就是这 4 个角点：`True` 返回坐标，`False` 返回这些点在轮廓中的索引 [3, 2, 1, 0]。


In [ ]:
# ===== 例 12.20 returnPoints 两种取值 =====
import cv2
o = shapes_img()                                  # 原书：contours.bmp
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_TREE,
                                       cv2.CHAIN_APPROX_SIMPLE)
cnt = [c for c in contours if cv2.boundingRect(c) == (25, 130, 71, 71)][0]   # 方块轮廓
hull = cv2.convexHull(cnt)                        # 默认 returnPoints=True：返回坐标
print("returnPoints 为默认值True时返回值hull的值：\n", hull)
hull2 = cv2.convexHull(cnt, returnPoints=False)   # 返回索引
print("returnPoints 为False时返回值hull的值：\n", hull2)


### 例 12.21 · 获取轮廓的凸包并绘制

凹口顶点 (150,150) 被排除在外——凸包只保留最外圈的 4 个角点，用 `cv2.polylines(o, [hull], True, (0,255,0), 2)` 把角点连成闭合折线（True 表示闭合）。


In [ ]:
# ===== 例 12.21 获取并绘制凸包 =====
import cv2
o = hand_img()                                    # 原书：hand.bmp（手掌）
cv2.imshow("original", o)
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
hull = cv2.convexHull(contours[0])                # 得到凸包角点
print('凸包角点 =', hull.reshape(-1, 2).tolist())
cv2.polylines(o, [hull], True, (0, 255, 0), 2)    # True：闭合折线
cv2.imshow("result", o)
cv2.waitKey()
cv2.destroyAllWindows()


### 12.5.2 凸缺陷：convexityDefects

`convexityDefects = cv2.convexityDefects(contour, convexhull)`：返回的数组每行包含 `[起点, 终点, 轮廓上距离凸包最远的点, 最远点到凸包的近似距离]`，**前三个值是轮廓点的索引**，需要回到轮廓点集里取坐标。

**注意**：第二个参数必须是"角点索引"形式的凸包，即查找凸包时要写 `cv2.convexHull(cnt, returnPoints=False)`，传坐标形式的凸包会得到错误结果或报错；距离字段是乘以 256 的定点数，取像素距离要除以 256。


### 例 12.22 · 计算并显示凸缺陷

起点与终点用红线连线、最远点画蓝色实心圆。本例只有一条缺陷：距离字段 25600 / 256 = 100.00，恰是凹口顶点 (150,150) 到凸包边 y=50 的垂直距离，手算吻合。


In [ ]:
# ===== 例 12.22 凸缺陷 =====
import cv2
img = hand_img()                                  # 原书：hand.bmp
cv2.imshow('original', img)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, 0)    # 0 等价于 cv2.THRESH_BINARY
contours, hierarchy = cv2.findContours(binary, cv2.RETR_TREE,
                                       cv2.CHAIN_APPROX_SIMPLE)
cnt = contours[0]
hull = cv2.convexHull(cnt, returnPoints=False)    # 关键：索引形式
defects = cv2.convexityDefects(cnt, hull)
print("defects=\n", defects)
for i in range(defects.shape[0]):
    s, e, f, d = defects[i, 0]                    # 起点、终点、最远点、距离
    start = tuple(cnt[s][0].tolist())             # 索引 → 坐标
    end = tuple(cnt[e][0].tolist())
    far = tuple(cnt[f][0].tolist())
    cv2.line(img, start, end, [0, 0, 255], 2)     # 起点到终点连线
    cv2.circle(img, far, 5, [255, 0, 0], -1)      # 最远点画实心圆
cv2.imshow('result', img)
cv2.waitKey(0)
cv2.destroyAllWindows()


### 12.5.3 几何学测试

**1. 测试轮廓是否是凸形的**：`retval = cv2.isContourConvex(contour)`，retval 为布尔值，True 表示轮廓是凸形的。

**2. 点到轮廓的距离**：`retval = cv2.pointPolygonTest(contour, pt, measureDist)`
- `measureDist=True`：计算点到轮廓的最短距离——点在轮廓**外部**返回负数、**轮廓上**返回 0、**内部**返回正数；
- `measureDist=False`：不计算距离，只返回 -1、0、1 三者之一，表示外部、轮廓上、内部。


### 例 12.23 · 判断凸包与逼近多边形是否为凸形

凸包必然是凸的（True）；epsilon=1% 周长的逼近多边形保留了 5 个顶点、凹口还没被"填平"，不是凸形（False）。凸包用绿色 polylines 画、逼近多边形用红色 drawContours 画。


In [ ]:
# ===== 例 12.23 isContourConvex =====
import cv2
o = hand_img()                                    # 原书：hand.bmp
cv2.imshow("original", o)
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
# ---------- 凸包 ----------
image1 = o.copy()
hull = cv2.convexHull(contours[0])
cv2.polylines(image1, [hull], True, (0, 255, 0), 2)
print("使用函数cv2.convexHull()构造的多边形是否是凸形的：",
      cv2.isContourConvex(hull))
cv2.imshow("result1", image1)
# ---------- 逼近多边形 ----------
image2 = o.copy()
epsilon = 0.01*cv2.arcLength(contours[0], True)
approx = cv2.approxPolyDP(contours[0], epsilon, True)
image2 = cv2.drawContours(image2, [approx], 0, (0, 0, 255), 2)
print("使用函数cv2.approxPolyDP()构造的多边形是否是凸形的：",
      cv2.isContourConvex(approx))
cv2.imshow("result2", image2)
cv2.waitKey()
cv2.destroyAllWindows()


### 例 12.24 · measureDist=True：计算三个点到轮廓的最短距离

内部点 A 返回 +50.0、外部点 B 返回 -50.0、顶点上的点 C 返回 0（打印成 -0.0 是浮点表示问题）。距离数值可以手算核对：凸包边界是 x=50、x=250、y=50、y=250，点 (100,200) 到最近边界恰好 50。


In [ ]:
# ===== 例 12.24 测距离 =====
import cv2
o = hand_img()                                    # 原书：cs.bmp
cv2.imshow("original", o)
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
hull = cv2.convexHull(contours[0])
distA = cv2.pointPolygonTest(hull, (100, 200), True)    # 内部点 A
print("distA=", distA)
distB = cv2.pointPolygonTest(hull, (300, 150), True)    # 外部点 B
print("distB=", distB)
distC = cv2.pointPolygonTest(hull, (50, 50), True)      # 顶点上的点 C
print("distC=", distC)
cv2.waitKey()
cv2.destroyAllWindows()


### 例 12.25 · measureDist=False：只判断位置关系

返回 1.0 / -1.0 / 0.0 表示内部、外部、轮廓上；不需要具体距离时用 False 更快。


In [ ]:
# ===== 例 12.25 只判位置 =====
import cv2
o = hand_img()                                    # 原书：cs.bmp
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
hull = cv2.convexHull(contours[0])
print("内部点A=", cv2.pointPolygonTest(hull, (100, 200), False))
print("外部点B=", cv2.pointPolygonTest(hull, (300, 150), False))
print("边上点C=", cv2.pointPolygonTest(hull, (50, 50), False))


## 12.6 利用形状场景算法比较轮廓

用矩比较形状很有效，但还有更有效的方法。从 OpenCV 3 开始，专有模块 shape 提供形状场景算法（shape scene），以"距离"作为形状比较的度量：两个形状一模一样时距离为零，差异越大距离越大。距离提取算子构造好后，统一用 `sd.computeDistance(contour1, contour2)` 计算两个轮廓的距离。


### 例 12.26 · 形状场景距离：createShapeContextDistanceExtractor

内部采用"形状上下文算法"（shape contexts）：在每个点上附加一个描述符，捕获其余点相对于它的分布特征，提供全局鉴别特征。相同对象距离为零，相似对象距离接近零，不相似对象距离明显更大——具体数值与对象、点数有关，规律一致。


In [ ]:
# ===== 例 12.26 形状场景距离 =====
import cv2
gray1 = cv2.cvtColor(quad_img((100, 100)), cv2.COLOR_BGR2GRAY)   # 原书 cs.bmp
gray2 = cv2.cvtColor(quad_img((130, 140)), cv2.COLOR_BGR2GRAY)   # 原书 cs3.bmp（移位后的自身）
gray3 = cv2.cvtColor(circ_img(), cv2.COLOR_BGR2GRAY)             # 原书 hand.bmp（不相似对象）
ret, b1 = cv2.threshold(gray1, 127, 255, cv2.THRESH_BINARY)
ret, b2 = cv2.threshold(gray2, 127, 255, cv2.THRESH_BINARY)
ret, b3 = cv2.threshold(gray3, 127, 255, cv2.THRESH_BINARY)
contours1, hierarchy = cv2.findContours(b1, cv2.RETR_LIST,
                                        cv2.CHAIN_APPROX_SIMPLE)
contours2, hierarchy = cv2.findContours(b2, cv2.RETR_LIST,
                                        cv2.CHAIN_APPROX_SIMPLE)
contours3, hierarchy = cv2.findContours(b3, cv2.RETR_LIST,
                                        cv2.CHAIN_APPROX_SIMPLE)
cnt1, cnt2, cnt3 = contours1[0], contours2[0], contours3[0]
# ---------- 构造距离提取算子并计算 ----------
sd = cv2.createShapeContextDistanceExtractor()
print("与自身的距离d1=", sd.computeDistance(cnt1, cnt1))
print("与移位后自身的距离d2=", sd.computeDistance(cnt1, cnt2))
print("与不相似对象的距离d3=", sd.computeDistance(cnt1, cnt3))


### 例 12.27 · Hausdorff 距离：createHausdorffDistanceExtractor

Hausdorff 距离：对 A 内每点找其到 B 的最短距离、取最大值，对 B 同样处理，再取两个最大值中的较大者。与 Hu 矩、形状场景距离不同，它**不具平移、缩放不变性**——同一形状移远一点距离立刻变大，比较形状前需要先把对象对齐。


In [ ]:
# ===== 例 12.27 Hausdorff 距离 =====
import cv2
gray1 = cv2.cvtColor(quad_img((100, 100)), cv2.COLOR_BGR2GRAY)   # 原书 cs.bmp
gray2 = cv2.cvtColor(quad_img((130, 140)), cv2.COLOR_BGR2GRAY)   # 移位后的自身
gray3 = cv2.cvtColor(circ_img(), cv2.COLOR_BGR2GRAY)             # 不相似对象
ret, b1 = cv2.threshold(gray1, 127, 255, cv2.THRESH_BINARY)
ret, b2 = cv2.threshold(gray2, 127, 255, cv2.THRESH_BINARY)
ret, b3 = cv2.threshold(gray3, 127, 255, cv2.THRESH_BINARY)
contours1, hierarchy = cv2.findContours(b1, cv2.RETR_LIST,
                                        cv2.CHAIN_APPROX_SIMPLE)
contours2, hierarchy = cv2.findContours(b2, cv2.RETR_LIST,
                                        cv2.CHAIN_APPROX_SIMPLE)
contours3, hierarchy = cv2.findContours(b3, cv2.RETR_LIST,
                                        cv2.CHAIN_APPROX_SIMPLE)
cnt1, cnt2, cnt3 = contours1[0], contours2[0], contours3[0]
hd = cv2.createHausdorffDistanceExtractor()
print("与自身图像的Hausdorff距离d1=", hd.computeDistance(cnt1, cnt1))
print("与移位后自身的Hausdorff距离d2=", hd.computeDistance(cnt1, cnt2))
print("与不相似对象的Hausdorff距离d3=", hd.computeDistance(cnt1, cnt3))


## 12.7 轮廓的特征值

轮廓自身的属性与轮廓所包围对象的特征对描述图像很有意义：宽高比、Extent、Solidity、等效直径、方向、掩模与像素点、最值、均值、极点——本节的特征值都可以由前面学过的函数组合出来。


### 例 12.28 · 宽高比 AspectRatio

`aspectRatio = float(w)/h`，w、h 来自 `cv2.boundingRect`——描述轮廓"扁"的程度（原书把该术语误拼为 AspectRation，正确拼写是 AspectRatio）。


In [ ]:
# ===== 例 12.28 宽高比 =====
import cv2
o = quad_img()
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)        # 原书用 100×60 的矩形轮廓 cc.bmp
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
x, y, w, h = cv2.boundingRect(contours[0])
cv2.rectangle(o, (x, y), (x+w, y+h), (255, 255, 255), 3)
aspectRatio = float(w)/h
print(aspectRatio)


### 例 12.29 · Extent：轮廓面积与矩形边界面积之比

`extend = float(cntArea)/rectArea`，衡量对象"填满"包围框的程度（100×60 矩形轮廓实测 0.9735，缺的一圈来自边界像素）。


In [ ]:
# ===== 例 12.29 Extent =====
import cv2
o = quad_img()
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
x, y, w, h = cv2.boundingRect(contours[0])
cv2.drawContours(o, contours[0], -1, (0, 0, 255), 3)
cv2.rectangle(o, (x, y), (x+w, y+h), (255, 0, 0), 3)
rectArea = w*h                                    # 矩形边界面积
cntArea = cv2.contourArea(contours[0])            # 轮廓面积
extend = float(cntArea)/rectArea
print(extend)


### 例 12.30 · Solidity：轮廓面积与凸包面积之比

`solidity = float(cntArea)/hullArea`，衡量对象被自己的凸包"填满"的程度：凸的方块该值为 1.0；带凹口的五边形实测 0.75（手算完全吻合：30000/40000）。


In [ ]:
# ===== 例 12.30 Solidity =====
import cv2
o = hand_img()                                    # 原书：hand.bmp
cv2.imshow("original", o)
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(o, contours[0], -1, (0, 0, 255), 3)
cntArea = cv2.contourArea(contours[0])            # 轮廓面积
hull = cv2.convexHull(contours[0])
hullArea = cv2.contourArea(hull)                  # 凸包面积
cv2.polylines(o, [hull], True, (0, 255, 0), 2)
solidity = float(cntArea)/hullArea
print(solidity)
cv2.imshow("result", o)
cv2.waitKey()
cv2.destroyAllWindows()


### 例 12.31 · 等效直径 Equivalent Diameter

`equiDiameter = np.sqrt(4*cntArea/np.pi)`：与轮廓面积相等的圆形的直径，并用 `cv2.circle` 画出这个等面积的圆。


In [ ]:
# ===== 例 12.31 等效直径 =====
import cv2
import numpy as np
o = quad_img()
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
cntArea = cv2.contourArea(contours[0])
equiDiameter = np.sqrt(4*cntArea/np.pi)           # sqrt(4×面积/π)
print(equiDiameter)
cv2.circle(o, (100, 100), int(equiDiameter/2), (0, 0, 255), 3)   # 等面积的圆
cv2.imshow("result", o)
cv2.waitKey()
cv2.destroyAllWindows()


### 例 12.32 · 方向：fitEllipse 的三个返回值形式

`(x, y), (MA, ma), angle = cv2.fitEllipse(cnt)`：拆开后直接拿到椭圆中心 (x, y)、水平/垂直方向轴长 (MA, ma)、旋转角度 angle，与单个返回值形式内容相同。


In [ ]:
# ===== 例 12.32 方向 =====
import cv2
o = quad_img()
cv2.imshow("original", o)
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
ellipse = cv2.fitEllipse(contours[0])             # 单个返回值
retval = cv2.fitEllipse(contours[0])
print("单个返回值形式：\n", retval)
(x, y), (MA, ma), angle = cv2.fitEllipse(contours[0])   # 三个返回值
print("(x,y)=", (x, y))
print("(MA,ma)=", (MA, ma))
print("angle=", angle)
cv2.ellipse(o, ellipse, (0, 0, 255), 2)
cv2.imshow("result", o)
cv2.waitKey()
cv2.destroyAllWindows()


### 例 12.33 / 12.35 · 非零值位置的两种获取方式

`np.transpose(np.nonzero(a))` 得到（行, 列）即 (y, x) 顺序；`cv2.findNonZero(a)` 得到（列, 行）即 (x, y) 顺序——两组输出互为转置，同一个非零点 Numpy 给 (0,1)、OpenCV 给 (1,0)。（原书说 transpose 处理后得到 (x, y) 形式坐标，不准确。）


In [ ]:
# ===== 例 12.33/12.35 非零值位置：Numpy 与 OpenCV =====
import cv2
import numpy as np
np.random.seed(0)                                 # 固定随机种子便于复现
a = np.zeros((5, 5), dtype=np.uint8)
for times in range(10):                           # 随机把 10 个位置置 1
    i = np.random.randint(0, 5)
    j = np.random.randint(0, 5)
    a[i, j] = 1                                   # 重复位置只算一个非零点
print("a=\n", a)
loc_np = np.transpose(np.nonzero(a))              # Numpy：(行, 列)
print("a 内非零值的位置（行,列）:\n", loc_np)
loc_cv = cv2.findNonZero(a)                       # OpenCV：(列, 行)
print("a 内非零值的位置（列,行）:\n", loc_cv)


### 例 12.34 / 12.36 · 获取图像轮廓的像素点位置

空心轮廓（thickness=2）只有边界上的 836 个点；实心轮廓（thickness=-1）包含整块 5041 = 71×71 个像素。`np.nonzero` 返回形状 (N, 2)、顺序（行, 列）；`findNonZero` 返回 (N, 1, 2)、顺序（列, 行）。


In [ ]:
# ===== 例 12.34/12.36 轮廓像素点 =====
import cv2
import numpy as np
o = quad_img()                                    # 原书：cc.bmp
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
cnt = contours[0]
# ---------- 空心轮廓 ----------
mask1 = np.zeros(gray.shape, np.uint8)
cv2.drawContours(mask1, [cnt], 0, 255, 2)         # thickness=2：空心
pp1 = np.transpose(np.nonzero(mask1))             # Numpy 方式：(行, 列)
pp1c = cv2.findNonZero(mask1)                     # OpenCV 方式：(列, 行)
print("空心轮廓 np.nonzero 形状 =", pp1.shape, " 前 3 个:", pp1[:3].tolist())
print("空心轮廓 findNonZero 形状 =", pp1c.shape, " 前 3 个:", pp1c.reshape(-1, 2)[:3].tolist())
# ---------- 实心轮廓 ----------
mask2 = np.zeros(gray.shape, np.uint8)
cv2.drawContours(mask2, [cnt], 0, 255, -1)        # thickness=-1：实心
pp2 = np.transpose(np.nonzero(mask2))
print("实心轮廓形状 =", pp2.shape)
cv2.imshow("mask1", mask1)
cv2.imshow("mask2", mask2)
cv2.waitKey()
cv2.destroyAllWindows()


### 例 12.37 · minMaxLoc：掩模区域内的最值及位置

`min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(imgray, mask=mask)`：mask 限定只统计指定区域。位置是 **(x, y) 即（列, 行）** 格式——对应写入时的 `g[31, 21] = 10`（第 31 行第 21 列）。若想获取彩色图像的最值，需拆出各通道分别计算。


In [ ]:
# ===== 例 12.37 minMaxLoc =====
import cv2
import numpy as np
g = np.zeros((300, 400), np.uint8)                # 原书：ct.png（含字母的图）
cv2.rectangle(g, (20, 30), (119, 89), 100, -1)    # 灰度 100 的矩形
cv2.circle(g, (280, 120), 50, 200, -1)            # 灰度 200 的圆
g[31, 21] = 10                                    # 矩形内人为放置最暗点
g[88, 118] = 250                                  # 矩形内人为放置最亮点
contours, hierarchy = cv2.findContours(g, cv2.RETR_EXTERNAL,
                                       cv2.CHAIN_APPROX_SIMPLE)
cnt = [c for c in contours if cv2.boundingRect(c) == (20, 30, 100, 60)][0]
mask = np.zeros(g.shape, np.uint8)                # 掩模必须是单通道
mask = cv2.drawContours(mask, [cnt], -1, 255, -1) # 实心轮廓当掩模
minVal, maxVal, minLoc, maxLoc = cv2.minMaxLoc(g, mask=mask)
print("minVal=", minVal)
print("maxVal=", maxVal)
print("minLoc=", minLoc)
print("maxLoc=", maxLoc)


### 例 12.38 · mean：平均颜色及平均灰度

`mean_val = cv2.mean(im, mask=mask)`：按 B、G、R、A 四个通道各返回一个均值。灰度图只有第 1 个值有意义（其余通道为 0）；掩模必须是单通道。


In [ ]:
# ===== 例 12.38 mean =====
import cv2
import numpy as np
g = np.zeros((300, 400), np.uint8)                # 同例 12.37 的灰度图
cv2.rectangle(g, (20, 30), (119, 89), 100, -1)
cv2.circle(g, (280, 120), 50, 200, -1)
g[31, 21] = 10
g[88, 118] = 250
contours, hierarchy = cv2.findContours(g, cv2.RETR_EXTERNAL,
                                       cv2.CHAIN_APPROX_SIMPLE)
cnt = [c for c in contours if cv2.boundingRect(c) == (20, 30, 100, 60)][0]
mask = np.zeros(g.shape, np.uint8)                # mean 的掩模必须是单通道
cv2.drawContours(mask, [cnt], 0, (255, 255, 255), -1)
meanVal = cv2.mean(g, mask=mask)
print("meanVal=\n", meanVal)
meanValColor = cv2.mean(cv2.cvtColor(g, cv2.COLOR_GRAY2BGR), mask=mask)
print("meanValColor=\n", meanValColor)


### 例 12.39 · 极点：argmin / argmax

沿轮廓点集的 x（`cnt[:, :, 0]`）、y（`cnt[:, :, 1]`）两维取 argmin/argmax，得到最左、最右、最上、最下四个极值点，并用 putText 标注 A/B/C/D。同一方向上有多个候选时，argmin/argmax 返回**第一个**出现的点。


In [ ]:
# ===== 例 12.39 极点 =====
import cv2
import numpy as np
o = circ_img()                                    # 原书：cs.bmp
gray = cv2.cvtColor(o, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
contours, hierarchy = cv2.findContours(binary, cv2.RETR_LIST,
                                       cv2.CHAIN_APPROX_SIMPLE)
cnt = contours[0]
leftmost = tuple(cnt[cnt[:, :, 0].argmin()][0].tolist())
rightmost = tuple(cnt[cnt[:, :, 0].argmax()][0].tolist())
topmost = tuple(cnt[cnt[:, :, 1].argmin()][0].tolist())
bottommost = tuple(cnt[cnt[:, :, 1].argmax()][0].tolist())
print("leftmost=", leftmost)
print("rightmost=", rightmost)
print("topmost=", topmost)
print("bottommost=", bottommost)
font = cv2.FONT_HERSHEY_SIMPLEX
cv2.putText(o, 'A', leftmost, font, 1, (0, 0, 255), 2)     # 标注四个极点
cv2.putText(o, 'B', rightmost, font, 1, (0, 0, 255), 2)
cv2.putText(o, 'C', topmost, font, 1, (0, 0, 255), 2)
cv2.putText(o, 'D', bottommost, font, 1, (0, 0, 255), 2)
cv2.imshow("result", o)
cv2.waitKey()
cv2.destroyAllWindows()
